# 07 · Subgroup Analysis & Fairness Auditing

## What this notebook covers
Aggregate metrics hide performance gaps across subgroups. A model with AUC=0.85 overall might score 0.70 for a minority group and 0.90 for the majority. Subgroup analysis surfaces these gaps; fairness metrics quantify them in terms regulators and product teams care about.

## Why it matters
Regulators (EU AI Act, US model risk guidance), enterprise buyers, and increasingly users expect disaggregated performance reporting. A single aggregate AUC is no longer sufficient for any model that affects people.

## The fairness metrics landscape
| Metric | Measures | Satisfiable simultaneously? |
|--------|----------|----------------------------|
| **Demographic parity** | P(ŷ=1 | A=0) ≈ P(ŷ=1 | A=1) | Not with other fairness criteria |
| **Equal opportunity** | TPR_A=0 ≈ TPR_A=1 | Mutually exclusive with DP in general |
| **Equalised odds** | TPR and FPR equal across groups | Strictest; rarely achievable exactly |

Impossibility theorems (Chouldechova 2017, Kleinberg et al. 2017) prove you cannot simultaneously satisfy demographic parity, equal opportunity, and calibration except in degenerate cases. The choice of metric should follow from the application's harm model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Synthetic dataset with group attribute
n = 2000
X, y = make_classification(n_samples=n, n_features=20, n_informative=12, random_state=0)
# Group A: 70% of data (advantaged), Group B: 30%
group = np.where(np.random.rand(n) < 0.7, "A", "B")

# Introduce a performance gap: add noise to Group B features
X_noisy = X.copy()
X_noisy[group == "B"] += np.random.randn((group == "B").sum(), X.shape[1]) * 0.8

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X_noisy, y, group, test_size=0.3, random_state=0
)

clf = GradientBoostingClassifier(n_estimators=100, random_state=0)
clf.fit(X_tr, y_tr)
proba_te = clf.predict_proba(X_te)[:, 1]
pred_te  = clf.predict(X_te)

print(f"Overall AUC: {roc_auc_score(y_te, proba_te):.4f}")

In [ ]:
def slice_metrics(y_true, y_pred, y_prob, groups):
    """Per-slice accuracy, AUC, F1, TPR, FPR."""
    results = []
    for g in sorted(set(groups)):
        mask = np.array(groups) == g
        yt, yp, ypr = y_true[mask], y_pred[mask], y_prob[mask]
        tp = ((yt == 1) & (yp == 1)).sum()
        fp = ((yt == 0) & (yp == 1)).sum()
        fn = ((yt == 1) & (yp == 0)).sum()
        tn = ((yt == 0) & (yp == 0)).sum()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        results.append({
            "group"   : g,
            "n"       : mask.sum(),
            "accuracy": round(accuracy_score(yt, yp), 4),
            "auc"     : round(roc_auc_score(yt, ypr) if len(set(yt)) > 1 else float("nan"), 4),
            "f1"      : round(f1_score(yt, yp, zero_division=0), 4),
            "tpr"     : round(tpr, 4),
            "fpr"     : round(fpr, 4),
        })
    return pd.DataFrame(results)

report = slice_metrics(y_te, pred_te, proba_te, g_te)
print(report.to_string(index=False))

In [ ]:
def disparity_report(slice_df):
    """Compute demographic parity ratio and equal opportunity ΔTPR."""
    rows = slice_df.set_index("group")
    groups = list(rows.index)
    if len(groups) < 2:
        return {}
    g_a, g_b = groups[0], groups[1]

    pos_rate_a = rows.loc[g_a, "tpr"] * rows.loc[g_a, "n"] / slice_df["n"].sum()
    pos_rate_b = rows.loc[g_b, "tpr"] * rows.loc[g_b, "n"] / slice_df["n"].sum()

    dp_ratio = min(rows.loc[g_a, "auc"], rows.loc[g_b, "auc"]) / max(rows.loc[g_a, "auc"], rows.loc[g_b, "auc"])
    delta_tpr = abs(rows.loc[g_a, "tpr"] - rows.loc[g_b, "tpr"])
    delta_auc = abs(rows.loc[g_a, "auc"] - rows.loc[g_b, "auc"])

    return {
        "auc_ratio (min/max)"        : round(dp_ratio, 4),
        "equal_opportunity_ΔTPR"     : round(delta_tpr, 4),
        "delta_AUC"                  : round(delta_auc, 4),
        "disparate_impact_flag"      : dp_ratio < 0.80,  # 80% rule
    }

disp = disparity_report(report)
print("\nDisparity report:")
for k, v in disp.items():
    print(f"  {k:<35}: {v}")
print("\nNote: AUC ratio <0.80 triggers the 80% rule — common in US employment law / EU AI Act bias assessments.")

In [ ]:
# Visualise per-group AUC and TPR
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colours = ["steelblue", "tomato"]
groups_list = report["group"].tolist()

for ax, metric in zip(axes, ["auc", "tpr"]):
    bars = ax.bar(groups_list, report[metric], color=colours)
    ax.set_ylim(0, 1)
    ax.set_ylabel(metric.upper())
    ax.set_title(f"Per-group {metric.upper()}")
    for bar, val in zip(bars, report[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.3f}", ha="center", fontsize=10)
    ax.axhline(0.80, color="black", linestyle="--", linewidth=1, label="80% threshold")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/07_subgroup_analysis.png", dpi=120)
plt.show()